Get schedule generation data and create csv

In [ ]:
%cd /Users/aliceliusyncrowin/Projects/tata-schedule-fetch
import json, os
import importlib
with open("local.settings.json", "r") as f:
    os.environ.update(json.load(f).get("Values", {}))
from helpers.schedule_generation import db, extract_jsonb
import pandas as pd
import datetime


In [ ]:
importlib.reload(db)
conn = db.get_connection()
try: 
    db.create_structured_tata_schedule_table(conn)
# Get all rows of the schedule data in jsonb format from the tata_schedule table
    rows = db.get_schedule_jsonb_rows(conn, 2)
# get the required fields from the jsonb in each row

# add the fields into a new table tata_schedule_structured


except Exception as e:

    print(f"An error occurred: {e}")
finally:
    conn.close()


In [ ]:
for row in rows:
    # check num of rows
    print(f"row: {row['schedule_date_ist']} {row['time_retrieved']}") 

test_row = rows[0]

In [ ]:
importlib.reload(extract_jsonb)
# test one rows extraction
results = extract_jsonb.flatten_jsonb(test_row['time_retrieved'], test_row['schedule_data'])

In [ ]:
print(results)

In [ ]:
rows_to_insert = []
for row in rows:
    rows_to_insert.extend(extract_jsonb.flatten_jsonb(row['time_retrieved'], row['schedule_data']))


In [ ]:
importlib.reload(db)
conn = db.get_connection()
try:
    rows = db.get_schedule_jsonb_rows(conn)
    all_rows_to_insert = []
    for row in rows:
        all_rows_to_insert.extend(extract_jsonb.flatten_jsonb(row['time_retrieved'], row['schedule_data']))

    print(f"Total rows to insert: {len(all_rows_to_insert)}")
    
    db.create_structured_tata_schedule_table(conn)

        

    db.add_schedule_data(conn, all_rows_to_insert)
finally:
    conn.close()

In [ ]:
latest_row = rows[-1]

In [ ]:
latest_row_json = latest_row['schedule_data']
latest_row_json 